# Gold Pass 5 — LSTM Payment Sequence Encoder
**PulseGuard** · Home Credit Default Risk

Trains a 1-layer LSTM on raw installment sequences → 32-dim embedding (features 141–172).

**Inputs** (upload from local `outputs/data/`):
- `inst_sequences.npy`  — float32 (N×50×3): days_late, payment_ratio, log_amt_instalment
- `inst_sk_ids.npy`    — int32   (N,): SK_ID_CURR for each sequence row
- `gp5_splits.npz`    — train/val/test IDs + TARGET labels

**Outputs**:
- `lstm_encoder.pt`              — saved PyTorch state dict
- `gp5_embeddings_train.npy`     — float32 (N_train×32)
- `gp5_embeddings_val.npy`       — float32 (N_val×32)
- `gp5_embeddings_test.npy`      — float32 (N_test×32)
- `gp5_median_embedding.npy`     — float32 (32,) — serving imputation vector

**Runtime**: ~8–12 min on Colab T4 GPU.

---
### Setup: change runtime to GPU first
> Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 0. Install / imports ───────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. Upload data files from your local machine ───────────────────────────
from google.colab import files
print('Upload inst_sequences.npy, inst_sk_ids.npy, gp5_splits.npz')
uploaded = files.upload()   # select all three files at once
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# ── 2. Load arrays ─────────────────────────────────────────────────────────
sequences = np.load('inst_sequences.npy')   # (N, 50, 3)
sk_ids    = np.load('inst_sk_ids.npy')      # (N,)  int32
splits    = np.load('gp5_splits.npz')

print(f'sequences: {sequences.shape}  dtype: {sequences.dtype}')
print(f'sk_ids:    {sk_ids.shape}')
print(f'splits keys: {list(splits.keys())}')

# Build id→row index for sequence lookup
id2row = {int(sk): i for i, sk in enumerate(sk_ids)}

def get_seqs(ids):
    """Look up sequences for given SK_ID_CURR array; zero-pad if missing."""
    out = np.zeros((len(ids), 50, 3), dtype=np.float32)
    for j, sk in enumerate(ids):
        row = id2row.get(int(sk))
        if row is not None:
            out[j] = sequences[row]
    return out

train_ids = splits['train_ids']; y_train = splits['y_train'].astype(np.float32)
val_ids   = splits['val_ids'];   y_val   = splits['y_val'].astype(np.float32)
test_ids  = splits['test_ids'];  y_test  = splits['y_test'].astype(np.float32)

print(f'train: {len(train_ids):,}  val: {len(val_ids):,}  test: {len(test_ids):,}')
print(f'default rate — train: {y_train.mean():.3f}  val: {y_val.mean():.3f}')

X_train = get_seqs(train_ids)   # (N_train, 50, 3)
X_val   = get_seqs(val_ids)
X_test  = get_seqs(test_ids)
print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

In [ ]:
# ── 3. Dataset ──────────────────────────────────────────────────────────────
class InstDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)   # (N, 50, 3)
        self.y = torch.from_numpy(y)   # (N,)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

BATCH = 2048
train_dl = DataLoader(InstDataset(X_train, y_train), batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(InstDataset(X_val,   y_val),   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Batches — train: {len(train_dl)}  val: {len(val_dl)}')

In [ ]:
# ── 4. Model ────────────────────────────────────────────────────────────────
class LSTMEncoder(nn.Module):
    """
    1-layer LSTM → 32-dim embedding → binary classification head.
    At serve time we use the 32-dim output of the projection layer
    as features 141–172 for LightGBM.
    """
    def __init__(self, input_size=3, hidden=64, embed_dim=32, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True,
                            bidirectional=False)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(hidden, embed_dim)   # → 32-dim embedding
        self.head = nn.Linear(embed_dim, 1)         # classification head

    def embed(self, x):
        """Return 32-dim embedding (used at inference time)."""
        _, (h, _) = self.lstm(x)       # h: (1, B, hidden)
        h = h.squeeze(0)               # (B, hidden)
        h = self.drop(h)
        return torch.tanh(self.proj(h))  # (B, 32)  — bounded [-1,1]

    def forward(self, x):
        emb = self.embed(x)            # (B, 32)
        return self.head(emb).squeeze(-1)  # (B,)  logits

model = LSTMEncoder().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

In [ ]:
# ── 5. Training ──────────────────────────────────────────────────────────────
# Class imbalance: ~8% positives → weighted BCE
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-5)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

EPOCHS     = 15
best_auc   = 0.0
best_state = None
history    = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    # ── train ──
    model.train()
    train_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(yb)
    train_loss /= len(train_dl.dataset)

    # ── val ──
    model.eval()
    val_logits, val_labels = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            logits = model(xb.to(device)).cpu()
            val_logits.append(logits); val_labels.append(yb)
    val_probs = torch.sigmoid(torch.cat(val_logits)).numpy()
    val_y     = torch.cat(val_labels).numpy()
    val_auc   = roc_auc_score(val_y, val_probs)
    scheduler.step(1 - val_auc)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f}  val_AUC={val_auc:.4f}  lr={optimizer.param_groups[0]["lr"]:.2e}  {elapsed:.0f}s')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_auc': val_auc})

    if val_auc > best_auc:
        best_auc   = val_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  ✓ new best AUC={best_auc:.4f}')

print(f'\nBest validation AUC: {best_auc:.4f}')

In [ ]:
# ── 6. Restore best weights + save encoder ──────────────────────────────────
model.load_state_dict(best_state)
torch.save(best_state, 'lstm_encoder.pt')
print('Saved lstm_encoder.pt')

In [ ]:
# ── 7. Extract embeddings for all three splits ───────────────────────────────
def extract_embeddings(X_np, batch_size=4096):
    """Run model.embed() over a numpy array in batches. Returns (N, 32) float32."""
    model.eval()
    embs = []
    with torch.no_grad():
        for start in range(0, len(X_np), batch_size):
            xb = torch.from_numpy(X_np[start:start+batch_size]).to(device)
            embs.append(model.embed(xb).cpu().numpy())
    return np.concatenate(embs, axis=0).astype(np.float32)

print('Extracting train embeddings …')
emb_train = extract_embeddings(X_train)
print(f'  emb_train: {emb_train.shape}')

print('Extracting val embeddings …')
emb_val = extract_embeddings(X_val)
print(f'  emb_val:   {emb_val.shape}')

print('Extracting test embeddings …')
emb_test = extract_embeddings(X_test)
print(f'  emb_test:  {emb_test.shape}')

# Compute median embedding across training set — used as serving imputation
median_emb = np.median(emb_train, axis=0).astype(np.float32)  # (32,)
print(f'  median_emb: {median_emb.shape}  range [{median_emb.min():.3f}, {median_emb.max():.3f}]')

np.save('gp5_embeddings_train.npy', emb_train)
np.save('gp5_embeddings_val.npy',   emb_val)
np.save('gp5_embeddings_test.npy',  emb_test)
np.save('gp5_median_embedding.npy', median_emb)
print('Saved all 4 embedding files')

In [ ]:
# ── 8. Sanity check: standalone LSTM AUC on test set ─────────────────────────
from sklearn.linear_model import LogisticRegression

# Quick logistic probe on top of embeddings
lr = LogisticRegression(max_iter=500, C=0.1).fit(emb_train, y_train.astype(int))
probe_auc = roc_auc_score(y_test, lr.predict_proba(emb_test)[:, 1])
print(f'Logistic probe AUC (emb_train → emb_test): {probe_auc:.4f}')

# Raw LSTM AUC on test
test_dl = DataLoader(InstDataset(X_test, y_test), batch_size=4096, shuffle=False)
model.eval()
test_logits = []
with torch.no_grad():
    for xb, _ in test_dl:
        test_logits.append(model(xb.to(device)).cpu())
test_probs = torch.sigmoid(torch.cat(test_logits)).numpy()
lstm_auc = roc_auc_score(y_test, test_probs)
print(f'LSTM standalone test AUC:                   {lstm_auc:.4f}')
print()
print('Expected range for LSTM-only: 0.60–0.68')
print('LightGBM champion baseline:   0.7769')
print('GP5 goal: AUC > 0.7769 after augmenting LightGBM with 32 LSTM features')

In [ ]:
# ── 9. Download all outputs ──────────────────────────────────────────────────
from google.colab import files
for fname in ['lstm_encoder.pt',
              'gp5_embeddings_train.npy',
              'gp5_embeddings_val.npy',
              'gp5_embeddings_test.npy',
              'gp5_median_embedding.npy']:
    files.download(fname)
    print(f'Downloaded {fname}')

print()
print('Copy all 5 files into: outputs/models/ in your pulseguard repo')
print('Then run: python scripts/gp5_augment_and_retrain.py')

## What to do with the outputs

1. Move the 5 downloaded files into `outputs/models/` in your local `pulseguard/` repo
2. Run `python scripts/gp5_augment_and_retrain.py`  (we'll write this next)
3. The script will augment X_train/val/test with the 32 LSTM embedding columns (141–172)
   and run a new LightGBM tournament
4. Report AUC vs 0.7769 baseline — if better, new champion; if not, document it honestly